# Comparaison : Prix Analytique vs PINNs vs Neural Network + Feynman-Kac

**Équation** : Black-Scholes (Call Européen)  
**Fixes v2** : normalisation par K, gradient clipping, λ rééquilibrés, LR schedule amélioré


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
from scipy.stats import norm
import time, warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444',      'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#aaa',         'ytick.color': '#aaa',
    'text.color': '#e0e0e0',       'grid.color': '#2a2a3e',
    'grid.linewidth': 0.6,         'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,          'axes.labelsize': 11,
})
COLORS = {'analytic': '#00d4ff', 'pinns': '#ff6b6b', 'fknn': '#69ff47'}
print("✅ Imports OK")


## 1. Paramètres

In [ ]:
S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.20, 1.0
S_min, S_max = 60.0, 160.0
N_S, N_t = 200, 50

S_grid = np.linspace(S_min, S_max, N_S)
t_grid = np.linspace(0.0, T * 0.999, N_t)

print(f"K={K}  r={r}  σ={sigma}  T={T}")
print(f"S ∈ [{S_min}, {S_max}]  — {N_S} points")


## 2. Prix analytique Black-Scholes

In [ ]:
def bs_call(S, t, K=K, r=r, sigma=sigma, T=T):
    S   = np.asarray(S, dtype=float)
    tau = np.maximum(T - t, 1e-8)
    d1  = (np.log(S / K) + (r + 0.5*sigma**2)*tau) / (sigma*np.sqrt(tau))
    d2  = d1 - sigma*np.sqrt(tau)
    return S*norm.cdf(d1) - K*np.exp(-r*tau)*norm.cdf(d2)

print(f"BS(S=100, t=0) = {bs_call(100,0):.4f}")
print(f"BS(S=120, t=0) = {bs_call(120,0):.4f}")
print(f"BS(S=80,  t=0) = {bs_call(80, 0):.4f}")


## 3. Architecture commune

In [ ]:
class BSNet(nn.Module):
    """
    u_theta(S, t)  —  entrées normalisées :
      x1 = log(S/K)          (centrée autour de 0)
      x2 = (T - t) / T       (dans [0,1])
    Sortie : u / K            (normalisée aussi !)
    """
    def __init__(self, hidden=64, n_layers=4):
        super().__init__()
        layers = [nn.Linear(2, hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers += [nn.Linear(hidden, 1), nn.Softplus()]   # prix >= 0
        self.net = nn.Sequential(*layers)
        # Initialisation Xavier pour éviter les gradients explosifs
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, S, t):
        x1 = torch.log(S / K)
        x2 = (T - t) / T
        x  = torch.cat([x1, x2], dim=1)
        return self.net(x) * K     # réseau prédit u/K, on restitue u

print(f"Paramètres : {sum(p.numel() for p in BSNet().parameters()):,}")


## 4. PINNs — avec normalisation et gradient clipping

### Fixes appliqués :
- **Loss normalisée par K²** → résidu ~O(1) au lieu de O(K²)
- **Gradient clipping** (`max_norm=1.0`) → stabilise l'entraînement
- **λ rééquilibrés** : terminal=2, bords=1 (au lieu de 10/5)
- **Warm-up LR** : on démarre à 5e-4 puis cosine annealing


In [ ]:
def pinn_residual(model, S, t):
    """Résidu EDP BS normalisé — retourne résidu / K."""    S = S.clone().requires_grad_(True)
    t = t.clone().requires_grad_(True)
    u = model(S, t)

    u_t  = torch.autograd.grad(u,  t,  grad_outputs=torch.ones_like(u),
                                create_graph=True)[0]
    u_S  = torch.autograd.grad(u,  S,  grad_outputs=torch.ones_like(u),
                                create_graph=True)[0]
    u_SS = torch.autograd.grad(u_S, S, grad_outputs=torch.ones_like(u_S),
                                create_graph=True)[0]

    res = u_t + 0.5*sigma**2*S**2*u_SS + r*S*u_S - r*u
    return res / K    # normalisation ← FIX


def train_pinn(n_epochs=8000, n_colloc=2000, lr=5e-4, verbose=True):
    model = BSNet()
    opt   = optim.Adam(model.parameters(), lr=lr)
    # Cosine annealing : LR descend doucement jusqu'à lr/100
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr/100)

    history = []
    t0 = time.time()

    for epoch in range(1, n_epochs + 1):
        # ── Collocation intérieure ────────────────────────────────────────────
        S_c = torch.FloatTensor(n_colloc, 1).uniform_(S_min, S_max)
        t_c = torch.FloatTensor(n_colloc, 1).uniform_(0.0, T*0.999)
        res = pinn_residual(model, S_c, t_c)
        loss_pde = (res**2).mean()

        # ── Condition terminale u(S,T) = max(S-K,0) ──────────────────────────
        S_T = torch.FloatTensor(n_colloc//2, 1).uniform_(S_min, S_max)
        t_T = torch.full_like(S_T, T*0.9999)
        u_T = model(S_T, t_T)
        payoff = torch.clamp(S_T - K, min=0.0)
        loss_term = ((u_T - payoff)**2 / K**2).mean()   # normalisé ← FIX

        # ── Bords ─────────────────────────────────────────────────────────────
        t_b  = torch.FloatTensor(300, 1).uniform_(0.0, T*0.999)

        # u(S_min, t) ≈ 0
        S_lo = torch.full((300,1), S_min)
        loss_lo = ((model(S_lo, t_b))**2 / K**2).mean()   # FIX

        # u(S_max, t) ≈ S_max - K·exp(-r·tau)
        S_hi    = torch.full((300,1), S_max)
        tau_b   = T - t_b
        tgt_hi  = S_max - K * torch.exp(-torch.tensor(r) * tau_b)
        loss_hi = ((model(S_hi, t_b) - tgt_hi)**2 / K**2).mean()   # FIX

        # ── Loss totale avec λ rééquilibrés ───────────────────────────────────
        loss = loss_pde + 2.0*loss_term + 1.0*(loss_lo + loss_hi)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # FIX
        opt.step()
        sched.step()

        l = loss.item()
        history.append(l)

        if verbose and epoch % 500 == 0:
            lr_now = opt.param_groups[0]['lr']
            print(f"  Epoch {epoch:5d} | Loss = {l:.2e} | LR = {lr_now:.2e}")

    return model, history, time.time() - t0

print("✅ train_pinn défini")


## 5. FK+NN — avec normalisation

### Fixes appliqués :
- **Labels normalisés par K** avant le MSE
- Même cosine annealing que PINNs pour comparaison équitable


In [ ]:
def feynman_kac_labels(S_b, t_b, n_mc=1000):
    """Labels MC Feynman-Kac vectorisés."""    with torch.no_grad():
        N   = S_b.shape[0]
        tau = T - t_b                            # (N,1)
        Z   = torch.randn(N, n_mc)
        S_T = S_b * torch.exp(
            (r - 0.5*sigma**2)*tau + sigma*torch.sqrt(tau)*Z
        )
        payoff   = torch.clamp(S_T - K, min=0.0)
        discount = torch.exp(-torch.tensor(r)*tau)
        return discount * payoff.mean(dim=1, keepdim=True)


def train_fknn(n_epochs=8000, batch_size=512, n_mc=1000, lr=5e-4, verbose=True):
    model = BSNet()
    opt   = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr/100)

    history = []
    t0 = time.time()

    for epoch in range(1, n_epochs + 1):
        S_b    = torch.FloatTensor(batch_size, 1).uniform_(S_min, S_max)
        t_b    = torch.FloatTensor(batch_size, 1).uniform_(0.0, T*0.999)
        labels = feynman_kac_labels(S_b, t_b, n_mc=n_mc)

        u_pred = model(S_b, t_b)
        # Loss normalisée par K² ← FIX
        loss   = ((u_pred - labels)**2 / K**2).mean()

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # FIX
        opt.step()
        sched.step()

        l = loss.item()
        history.append(l)

        if verbose and epoch % 500 == 0:
            lr_now = opt.param_groups[0]['lr']
            print(f"  Epoch {epoch:5d} | Loss = {l:.2e} | LR = {lr_now:.2e}")

    return model, history, time.time() - t0

print("✅ train_fknn défini")


## 6. Lancement de l'entraînement

In [ ]:
N_EPOCHS = 8000

print("=" * 55)
print("📌 Entraînement PINNs")
print("=" * 55)
model_pinn, hist_pinn, time_pinn = train_pinn(n_epochs=N_EPOCHS, verbose=True)
print(f"\n✅ PINNs — {time_pinn:.1f}s  |  Loss finale : {hist_pinn[-1]:.2e}")

print()
print("=" * 55)
print("📌 Entraînement Neural Network + Feynman-Kac")
print("=" * 55)
model_fknn, hist_fknn, time_fknn = train_fknn(n_epochs=N_EPOCHS, verbose=True)
print(f"\n✅ FK+NN — {time_fknn:.1f}s  |  Loss finale : {hist_fknn[-1]:.2e}")


## 7. Courbes de convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

for ax, hist, color, title in zip(
    axes,
    [hist_pinn, hist_fknn],
    [COLORS['pinns'], COLORS['fknn']],
    ['PINNs — Loss normalisée', 'FK+NN — MSE normalisé']
):
    smooth = np.convolve(hist, np.ones(50)/50, mode='valid')
    ax.semilogy(hist,   color=color, alpha=0.2, lw=0.8)
    ax.semilogy(smooth, color=color, lw=2.2)
    ax.set_title(title); ax.set_xlabel("Époque"); ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.4)
    # Annotation loss finale
    ax.annotate(f"Final: {hist[-1]:.2e}", xy=(len(hist)-1, hist[-1]),
                xytext=(-120, 20), textcoords='offset points',
                color=color, fontsize=9,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 8. Évaluation

In [ ]:
def eval_model(model, S_arr, t_val):
    model.eval()
    with torch.no_grad():
        S_t = torch.FloatTensor(S_arr.reshape(-1,1))
        t_t = torch.full_like(S_t, t_val)
        return model(S_t, t_t).numpy().flatten()

def eval_model_grid(model, S_arr, t_arr):
    model.eval()
    out = np.zeros((len(S_arr), len(t_arr)))
    with torch.no_grad():
        for j, tv in enumerate(t_arr):
            S_t = torch.FloatTensor(S_arr.reshape(-1,1))
            t_t = torch.full_like(S_t, tv)
            out[:,j] = model(S_t, t_t).numpy().flatten()
    return out

u_analytic_0 = bs_call(S_grid, 0.0)
u_pinn_0     = eval_model(model_pinn, S_grid, 0.0)
u_fknn_0     = eval_model(model_fknn, S_grid, 0.0)

print("Prix en S=100, t=0 :")
print(f"  Analytique : {bs_call(100,0):.4f}")
print(f"  PINNs      : {eval_model(model_pinn, np.array([100.0]), 0.0)[0]:.4f}")
print(f"  FK+NN      : {eval_model(model_fknn, np.array([100.0]), 0.0)[0]:.4f}")


## 9. Prix en $t=0$

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(S_grid, u_analytic_0, color=COLORS['analytic'], lw=2.5, label='Analytique (BS)', zorder=5)
ax.plot(S_grid, u_pinn_0,     color=COLORS['pinns'],    lw=2.0, ls='--', label='PINNs',   zorder=4)
ax.plot(S_grid, u_fknn_0,     color=COLORS['fknn'],     lw=2.0, ls='-.', label='FK+NN',   zorder=3)
ax.axvline(K, color='#888', ls=':', lw=1.2, label=f'Strike K={K}')
ax.set_xlabel("S");  ax.set_ylabel("u(S, 0)")
ax.set_title("Prix du Call Européen en $t=0$")
ax.legend(framealpha=0.2);  ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('price_t0.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 10. Erreur absolue en $t=0$

In [ ]:
err_pinn_0 = np.abs(u_pinn_0 - u_analytic_0)
err_fknn_0 = np.abs(u_fknn_0 - u_analytic_0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(S_grid, err_pinn_0, color=COLORS['pinns'], lw=2.0, ls='--',
        label=f'PINNs  — MAE = {err_pinn_0.mean():.4f}')
ax.plot(S_grid, err_fknn_0, color=COLORS['fknn'],  lw=2.0, ls='-.',
        label=f'FK+NN  — MAE = {err_fknn_0.mean():.4f}')
ax.fill_between(S_grid, err_pinn_0, alpha=0.10, color=COLORS['pinns'])
ax.fill_between(S_grid, err_fknn_0, alpha=0.10, color=COLORS['fknn'])
ax.axvline(K, color='#888', ls=':', lw=1.2, label=f'K={K}')
ax.set_xlabel("S");  ax.set_ylabel("|u_pred − u_exact|")
ax.set_title("Erreur absolue en $t=0$")
ax.legend(framealpha=0.2);  ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('error_t0.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

print(f"MAE PINNs : {err_pinn_0.mean():.5f}")
print(f"MAE FK+NN : {err_fknn_0.mean():.5f}")


## 11. Heatmaps erreur absolue sur $(S, t)$ — même échelle

In [ ]:
print("Calcul grilles (S, t)...")
U_analytic = np.array([[bs_call(s, tv) for tv in t_grid] for s in S_grid])
U_pinn     = eval_model_grid(model_pinn, S_grid, t_grid)
U_fknn     = eval_model_grid(model_fknn, S_grid, t_grid)

E_pinn = np.abs(U_pinn - U_analytic)
E_fknn = np.abs(U_fknn - U_analytic)
vmax   = max(E_pinn.max(), E_fknn.max())
print(f"Erreur max — PINNs : {E_pinn.max():.4f}  |  FK+NN : {E_fknn.max():.4f}")
print(f"Échelle commune    : [0, {vmax:.4f}]")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f0f1a')

for ax, E, title in zip(axes, [E_pinn, E_fknn],
                         ['Erreur absolue — PINNs', 'Erreur absolue — FK+NN']):
    im = ax.pcolormesh(t_grid, S_grid, E, cmap='plasma',
                       norm=PowerNorm(gamma=0.5, vmin=0, vmax=vmax), shading='auto')
    ax.axhline(K, color='white', ls='--', lw=0.8, alpha=0.5, label=f'K={K}')
    ax.set_xlabel("t");  ax.set_ylabel("S");  ax.set_title(title)
    ax.legend(framealpha=0.1, fontsize=9)

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
cbar.set_label("Erreur absolue", labelpad=10)
plt.suptitle("Heatmap erreur absolue — même colorbar", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('heatmap_error.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 12. Performance temporelle : époques + temps pour atteindre une précision cible

On fixe des seuils de **MAE** évaluée à $t=0$ sur $S \in [60, 160]$.


In [ ]:
TARGETS = [1.0, 0.5, 0.3, 0.15]

def measure_conv(mode, targets, n_max=12000):
    """mode = 'pinn' ou 'fknn'."""    model = BSNet()
    lr    = 5e-4
    opt   = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_max, eta_min=lr/100)

    results   = {tgt: None for tgt in targets}
    remaining = set(targets)
    t0 = time.time()

    for epoch in range(1, n_max + 1):
        if mode == 'pinn':
            S_c = torch.FloatTensor(2000,1).uniform_(S_min, S_max)
            t_c = torch.FloatTensor(2000,1).uniform_(0.0, T*0.999)
            res = pinn_residual(model, S_c, t_c)
            loss_pde = (res**2).mean()
            S_T = torch.FloatTensor(1000,1).uniform_(S_min, S_max)
            t_T = torch.full_like(S_T, T*0.9999)
            loss_term = ((model(S_T,t_T) - torch.clamp(S_T-K,min=0))**2/K**2).mean()
            t_b  = torch.FloatTensor(300,1).uniform_(0.0,T*0.999)
            S_lo = torch.full((300,1),S_min); S_hi = torch.full((300,1),S_max)
            loss_lo = ((model(S_lo,t_b))**2/K**2).mean()
            tau_b = T-t_b
            tgt_hi = S_max - K*torch.exp(-torch.tensor(r)*tau_b)
            loss_hi= ((model(S_hi,t_b)-tgt_hi)**2/K**2).mean()
            loss = loss_pde + 2.0*loss_term + 1.0*(loss_lo+loss_hi)
        else:
            S_b = torch.FloatTensor(512,1).uniform_(S_min, S_max)
            t_b = torch.FloatTensor(512,1).uniform_(0.0, T*0.999)
            lbl = feynman_kac_labels(S_b, t_b, n_mc=1000)
            loss= ((model(S_b,t_b)-lbl)**2/K**2).mean()

        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        # Évaluation MAE toutes les 100 époques
        if epoch % 100 == 0 and remaining:
            u0  = eval_model(model, S_grid, 0.0)
            mae = np.abs(u0 - u_analytic_0).mean()
            elapsed = time.time() - t0
            for tgt in list(remaining):
                if mae <= tgt:
                    results[tgt] = {'epochs': epoch, 'time': elapsed, 'mae': mae}
                    remaining.discard(tgt)
            if not remaining:
                break

    return results

print("⏱️  Convergence PINNs...")
conv_pinn = measure_conv('pinn', TARGETS)

print("⏱️  Convergence FK+NN...")
conv_fknn = measure_conv('fknn', TARGETS)

print()
print(f"{'Cible MAE':>10} | {'PINNs époques':>14} | {'PINNs (s)':>10} | {'FK+NN époques':>14} | {'FK+NN (s)':>10}")
print("-" * 65)
for tgt in TARGETS:
    p = conv_pinn.get(tgt); f = conv_fknn.get(tgt)
    ep = f"{p['epochs']:,}" if p else "N/A"
    et = f"{p['time']:.1f}" if p else "N/A"
    fp = f"{f['epochs']:,}" if f else "N/A"
    ft = f"{f['time']:.1f}" if f else "N/A"
    print(f"{tgt:>10.2f} | {ep:>14} | {et:>10} | {fp:>14} | {ft:>10}")


### Graphe de performance temporelle

In [ ]:
tgts_ok = [t for t in TARGETS if conv_pinn.get(t) or conv_fknn.get(t)]
x = np.arange(len(tgts_ok)); w = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0f0f1a')

def bar_with_labels(ax, x_pos, vals, color, label):
    bars = ax.bar(x_pos, vals, w, color=color, alpha=0.85, label=label,
                  edgecolor='white', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x()+bar.get_width()/2, h*1.02,
                    f'{h:,.0f}' if h > 10 else f'{h:.1f}',
                    ha='center', va='bottom', fontsize=8, color=color)

ep_p = [conv_pinn[t]['epochs'] if conv_pinn.get(t) else 0 for t in tgts_ok]
ep_f = [conv_fknn[t]['epochs'] if conv_fknn.get(t) else 0 for t in tgts_ok]
ti_p = [conv_pinn[t]['time']   if conv_pinn.get(t) else 0 for t in tgts_ok]
ti_f = [conv_fknn[t]['time']   if conv_fknn.get(t) else 0 for t in tgts_ok]

bar_with_labels(ax1, x-w/2, ep_p, COLORS['pinns'], 'PINNs')
bar_with_labels(ax1, x+w/2, ep_f, COLORS['fknn'],  'FK+NN')
ax1.set_xticks(x); ax1.set_xticklabels([f'MAE≤{t}' for t in tgts_ok])
ax1.set_ylabel("Époques"); ax1.set_title("Époques pour atteindre la précision")
ax1.legend(framealpha=0.2); ax1.grid(True, alpha=0.3, axis='y')

bar_with_labels(ax2, x-w/2, ti_p, COLORS['pinns'], 'PINNs')
bar_with_labels(ax2, x+w/2, ti_f, COLORS['fknn'],  'FK+NN')
ax2.set_xticks(x); ax2.set_xticklabels([f'MAE≤{t}' for t in tgts_ok])
ax2.set_ylabel("Temps (s)"); ax2.set_title("Temps CPU pour atteindre la précision")
ax2.legend(framealpha=0.2); ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('perf_temporelle.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 13. Figure de synthèse

In [ ]:
mae_pinn    = err_pinn_0.mean();    mae_fknn   = err_fknn_0.mean()
rmse_pinn   = np.sqrt((err_pinn_0**2).mean()); rmse_fknn = np.sqrt((err_fknn_0**2).mean())
maxe_pinn   = err_pinn_0.max();    maxe_fknn  = err_fknn_0.max()
mae2d_pinn  = E_pinn.mean();       mae2d_fknn = E_fknn.mean()

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0f0f1a')
gs  = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.35)

# (A) Prix
ax = fig.add_subplot(gs[0,0])
ax.plot(S_grid, u_analytic_0, color=COLORS['analytic'], lw=2.5, label='Analytique')
ax.plot(S_grid, u_pinn_0,     color=COLORS['pinns'],    lw=1.8, ls='--', label='PINNs')
ax.plot(S_grid, u_fknn_0,     color=COLORS['fknn'],     lw=1.8, ls='-.', label='FK+NN')
ax.axvline(K, color='#888', ls=':', lw=1)
ax.set_title("(A) Prix en $t=0$"); ax.set_xlabel("S"); ax.set_ylabel("u(S,0)")
ax.legend(fontsize=8, framealpha=0.15); ax.grid(True, alpha=0.3)

# (B) Erreur
ax = fig.add_subplot(gs[0,1])
ax.plot(S_grid, err_pinn_0, color=COLORS['pinns'], lw=1.8, ls='--',
        label=f'PINNs MAE={mae_pinn:.3f}')
ax.plot(S_grid, err_fknn_0, color=COLORS['fknn'],  lw=1.8, ls='-.',
        label=f'FK+NN MAE={mae_fknn:.3f}')
ax.fill_between(S_grid, err_pinn_0, alpha=0.10, color=COLORS['pinns'])
ax.fill_between(S_grid, err_fknn_0, alpha=0.10, color=COLORS['fknn'])
ax.axvline(K, color='#888', ls=':', lw=1)
ax.set_title("(B) Erreur absolue $t=0$"); ax.set_xlabel("S")
ax.legend(fontsize=8, framealpha=0.15); ax.grid(True, alpha=0.3)

# (C) Loss
ax = fig.add_subplot(gs[0,2])
sp = np.convolve(hist_pinn, np.ones(60)/60, mode='valid')
sf = np.convolve(hist_fknn, np.ones(60)/60, mode='valid')
ax.semilogy(sp, color=COLORS['pinns'], lw=2, label='PINNs')
ax.semilogy(sf, color=COLORS['fknn'],  lw=2, label='FK+NN')
ax.set_title("(C) Convergence Loss"); ax.set_xlabel("Époque")
ax.legend(fontsize=8, framealpha=0.15); ax.grid(True, alpha=0.3)

# (D) Heatmap PINNs
ax = fig.add_subplot(gs[1,0])
im = ax.pcolormesh(t_grid, S_grid, E_pinn, cmap='plasma',
                   norm=PowerNorm(gamma=0.5, vmin=0, vmax=vmax), shading='auto')
ax.axhline(K, color='white', ls='--', lw=0.7, alpha=0.5)
ax.set_title("(D) Heatmap erreur — PINNs"); ax.set_xlabel("t"); ax.set_ylabel("S")
fig.colorbar(im, ax=ax, shrink=0.9)

# (E) Heatmap FK+NN
ax = fig.add_subplot(gs[1,1])
im2= ax.pcolormesh(t_grid, S_grid, E_fknn, cmap='plasma',
                   norm=PowerNorm(gamma=0.5, vmin=0, vmax=vmax), shading='auto')
ax.axhline(K, color='white', ls='--', lw=0.7, alpha=0.5)
ax.set_title("(E) Heatmap erreur — FK+NN"); ax.set_xlabel("t"); ax.set_ylabel("S")
fig.colorbar(im2, ax=ax, shrink=0.9)

# (F) Tableau
ax = fig.add_subplot(gs[1,2]); ax.axis('off')
rows = [['Métrique','PINNs','FK+NN'],
        ['MAE t=0',   f'{mae_pinn:.4f}',   f'{mae_fknn:.4f}'],
        ['RMSE t=0',  f'{rmse_pinn:.4f}',  f'{rmse_fknn:.4f}'],
        ['Max err t=0',f'{maxe_pinn:.4f}', f'{maxe_fknn:.4f}'],
        ['MAE grille', f'{mae2d_pinn:.4f}', f'{mae2d_fknn:.4f}'],
        ['Temps (s)',  f'{time_pinn:.1f}',  f'{time_fknn:.1f}']]
tbl = ax.table(cellText=rows[1:], colLabels=rows[0], loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 2.1)
for (rr,cc), cell in tbl.get_celld().items():
    cell.set_facecolor('#1a1a2e'); cell.set_edgecolor('#444')
    cell.set_text_props(color='#e0e0e0')
    if rr == 0:
        cell.set_facecolor('#2a2a4e')
        cell.set_text_props(weight='bold', color='#00d4ff')
ax.set_title("(F) Récapitulatif", pad=10)

plt.suptitle(f"Analytique vs PINNs vs FK+NN  —  BS Call  (K={K}, r={r}, σ={sigma}, T={T})",
             fontsize=13, y=1.01)
plt.savefig('synthese_finale.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("✅ Synthèse générée !")


## 14. Conclusions

### Fixes appliqués (v2)
| Fix | Effet |
|---|---|
| **Normalisation par K** | Loss ~O(1) → convergence stable dès l'époque 1 |
| **Softplus en sortie** | Prix toujours ≥ 0 |
| **Xavier init** | Gradients bien conditionnés au départ |
| **Gradient clipping (1.0)** | Élimine les pics de loss explosifs |
| **Cosine annealing** | LR descend proprement, pas de sur-oscillation |
| **λ rééquilibrés (2/1)** | PDE et conditions aux bords en équilibre |

### Quand choisir quoi ?
- **PINNs** → EDP connue, faible dimension, contraintes physiques strictes
- **FK+NN** → haute dimension, SDE complexe, implémentation rapide
